# Free cloud trainer for *Inside One Training Step*

This notebook starts the site's **real PyTorch training companion** on a free
Google Colab machine and links it to the hosted site through your own private
HTTPS tunnel.

## How to use it — three clicks

**1 · Run everything.** Click **Run all** in the toolbar above (shown in your
Google language — for example *Tümünü çalıştır*), then approve the standard
"not authored by Google" notice.

<img src="https://raw.githubusercontent.com/kocahmet1/3d-llm/main/notebooks/assets/step1-run-all-toolbar.png" alt="The Run all button in the Colab toolbar" height="52">

*…or press the round ▶ button on the first cell instead:*

<img src="https://raw.githubusercontent.com/kocahmet1/3d-llm/main/notebooks/assets/step2-run-cell.png" alt="The play button on the first cell" height="200">

**2 · Wait about a minute.** The first cell reports its progress as
`Step 1/5 … Step 5/5` while it installs the trainer and opens your tunnel.

**3 · Click the green connect button** that appears under the first cell. It
returns you to the site with your trainer already connected.

<img src="https://raw.githubusercontent.com/kocahmet1/3d-llm/main/notebooks/assets/step3-connect-button.png" alt="The green connect button printed below the first cell" height="150">

Keep this Colab tab open while you train. Closing it (or Colab idling out)
stops the trainer; your run's last received state stays visible on the site.

*Tip: for the fastest training, use **Runtime → Change runtime type → T4 GPU**
before running. CPU works too.*

**Privacy:** the text you upload on the site goes only to this Colab session of
yours, protected by a one-time token. Nothing is sent to the site's servers.


In [ ]:
# @title 1 · Start your trainer (Runtime → Run all) { display-mode: "form" }
site_url = "https://inside-one-training-step.vocabdeneme.chatgpt.site"  # @param {type:"string"}
repo_url = "https://github.com/kocahmet1/3d-llm.git"  # @param {type:"string"}
branch = "main"  # @param {type:"string"}

import json
import os
import re
import secrets
import subprocess
import sys
import time
import urllib.request
from pathlib import Path
from urllib.parse import quote, urlsplit

WORK = Path("/content")
REPO = WORK / "3d-llm"
PORT = 8765

site = site_url.strip().rstrip("/")
site_parts = urlsplit(site)
assert site_parts.scheme == "https" and site_parts.hostname, "site_url must be a full https:// address"
site_origin = f"{site_parts.scheme}://{site_parts.netloc}"

print("Step 1/5 — fetching the trainer code…")
if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=False)
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", branch, repo_url, str(REPO)],
        check=True,
    )

print("Step 2/5 — installing the trainer package…")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", str(REPO / "trainer")],
    check=True,
)

import torch

if torch.cuda.is_available():
    device_note = f"free GPU active ({torch.cuda.get_device_name(0)})"
else:
    device_note = (
        "no GPU on this runtime — CPU still works; for speed use "
        "Runtime → Change runtime type → T4 GPU, then Run all again"
    )
print(f"   PyTorch {torch.__version__} ready — {device_note}.")

print("Step 3/5 — opening your private HTTPS tunnel…")
cloudflared = WORK / "cloudflared"
if not cloudflared.exists():
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        cloudflared,
    )
    cloudflared.chmod(0o755)

tunnel_log_path = WORK / "cloudflared.log"
tunnel_log = open(tunnel_log_path, "w")
tunnel = subprocess.Popen(
    [str(cloudflared), "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT,
)

tunnel_url = None
for _ in range(120):
    match = re.search(
        r"https://[-a-z0-9]+\.trycloudflare\.com",
        tunnel_log_path.read_text(errors="ignore"),
    )
    if match:
        tunnel_url = match.group(0)
        break
    time.sleep(1)
assert tunnel_url, "the tunnel did not start in time — run this cell again"
tunnel_host = urlsplit(tunnel_url).hostname

print("Step 4/5 — starting the PyTorch training companion…")
token = secrets.token_urlsafe(24)
companion_env = dict(
    os.environ,
    CHAMBER_TRAINER_AUTH_TOKEN=token,
    CHAMBER_TRAINER_ALLOWED_ORIGINS=site_origin,
    CHAMBER_TRAINER_ALLOWED_HOSTS=tunnel_host,
)
companion_log_path = WORK / "companion.log"
companion_log = open(companion_log_path, "w")
companion = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "chamber_trainer",
        "serve",
        "--port",
        str(PORT),
        "--runs-dir",
        str(WORK / "chamber-runs"),
    ],
    cwd=str(REPO / "trainer"),
    env=companion_env,
    stdout=companion_log,
    stderr=subprocess.STDOUT,
)


def probe(base: str) -> bool:
    request = urllib.request.Request(
        base + "/health", headers={"Authorization": f"Bearer {token}"}
    )
    with urllib.request.urlopen(request, timeout=5) as response:
        return json.load(response).get("ok") is True


for _ in range(60):
    if companion.poll() is not None:
        print(companion_log_path.read_text(errors="ignore")[-2000:])
        raise SystemExit("the companion exited early — see its log above")
    try:
        if probe(f"http://127.0.0.1:{PORT}"):
            break
    except Exception:
        time.sleep(1)
else:
    raise SystemExit("the companion did not answer in time — run this cell again")

print("Step 5/5 — checking the tunnel end to end…")
for _ in range(60):
    try:
        if probe(tunnel_url):
            break
    except Exception:
        time.sleep(2)
else:
    raise SystemExit("the tunnel is not routable yet — run this cell again")

connect_link = (
    f"{site}/custom-training"
    f"#trainer={quote(tunnel_url, safe='')}"
    f"&trainerToken={quote(token, safe='')}"
)

from IPython.display import HTML, display

display(
    HTML(
        f'<div style="margin:14px 0;padding:18px;border:2px solid #2fbf8f;'
        f'border-radius:12px;background:#07231b;font-family:system-ui">'
        f'<div style="color:#8ff9c9;font-size:13px;letter-spacing:.08em;'
        f'font-weight:700">YOUR TRAINER IS LIVE</div>'
        f'<a href="{connect_link}" target="_blank" style="display:inline-block;'
        f'margin:10px 0;padding:12px 18px;border-radius:9px;background:#2fbf8f;'
        f'color:#04120d;font-weight:800;font-size:16px;text-decoration:none">'
        f'Click here to connect &rarr;</a>'
        f'<div style="color:#9fc0b5;font-size:12px">Keep this Colab tab open '
        f'while you train. If the button does not open, copy this link into the '
        f'site\u2019s \u201cpaste the connect link\u201d field:</div>'
        f'<code style="display:block;margin-top:6px;color:#d9fff0;font-size:11px;'
        f'word-break:break-all">{connect_link}</code></div>'
    )
)
print("Trainer ready. Run the next cell to watch live logs (optional).")


In [ ]:
# @title 2 · (Optional) Watch live trainer logs — also helps keep Colab awake
import time
from pathlib import Path

log_path = Path("/content/companion.log")
position = 0
print("Watching trainer logs — stop this cell any time; the trainer keeps running.")
try:
    while True:
        if log_path.exists():
            with log_path.open(errors="ignore") as handle:
                handle.seek(position)
                chunk = handle.read()
                position = handle.tell()
            if chunk:
                print(chunk, end="")
        time.sleep(2)
except KeyboardInterrupt:
    print("\nStopped watching; the trainer keeps running.")
